In [ ]:
import os
import zipfile
import requests

# 1. Downloading a manageable subset of PlantVillage (200MB)
url = "https://github.com/spMohanty/PlantVillage-Dataset/archive/refs/heads/master.zip"
print("Downloading mini-dataset... this will be much faster!")

r = requests.get(url)
with open("mini_plant.zip", "wb") as f:
    f.write(r.content)

# 2. Unzip
with zipfile.ZipFile("mini_plant.zip", "r") as zip_ref:
    zip_ref.extractall("plant_data")

print("✅ Mini-dataset ready!")

✅ Mini-dataset ready!


In [ ]:
import os
# Looking for the folder containing the actual images
base_path = "plant_data/PlantVillage-Dataset-master/raw/color"
if os.path.exists(base_path):
    classes = os.listdir(base_path)
    print(f"Total classes found: {len(classes)}")
    print(f"Sample classes: {classes[:5]}")

In [7]:
import os
import zipfile
import requests
import shutil

# 1. Download the file
url = "https://github.com/spMohanty/PlantVillage-Dataset/archive/refs/heads/master.zip"
print("Downloading...")
r = requests.get(url)
with open("mini_plant.zip", "wb") as f:
    f.write(r.content)

# 2. Clean up and Unzip
if os.path.exists("plant_dataset"):
    shutil.rmtree("plant_dataset")

print("Unzipping...")
with zipfile.ZipFile("mini_plant.zip", "r") as zip_ref:
    zip_ref.extractall("plant_dataset")

# 3. The "Detective" script to find your data
found_path = ""
for root, dirs, files in os.walk("plant_dataset"):
    if len(dirs) > 30:
        found_path = root
        break

if found_path:
    base_path = found_path
    classes = sorted(os.listdir(base_path))
    print(f"🎯 SUCCESS! Path found: {base_path}")
    print(f"Total classes: {len(classes)}")
else:
    print("❌ Still can't find the folders. Check the sidebar!")

Downloading...
Unzipping...
🎯 SUCCESS! Path found: plant_dataset/PlantVillage-Dataset-master/data_distribution_for_SVM/train
Total classes: 38


In [10]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split

# 1. Setup Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Data Preparation (Using base_path from your detective script)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Re-defining full_datasets right here so it's fresh in memory
full_datasets = datasets.ImageFolder(base_path, transform=transform)

# 3. Splitting
train_size = int(0.8 * len(full_datasets))
val_size = len(full_datasets) - train_size
train_dataset, val_dataset = random_split(full_datasets, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# 4. Model Surgery
model = models.mobilenet_v2(weights='DEFAULT')

for param in model.parameters():
    param.requires_grad = False

num_classes = len(full_datasets.classes)
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, num_classes)

model = model.to(device)

print(f"✅ Success! Data loaded and Model wired for {num_classes} classes.")
print(f"📦 Training images: {len(train_dataset)}")

✅ Success! Data loaded and Model wired for 38 classes.
📦 Training images: 7000


In [11]:
import torch.optim as optim

# 1. The Grading System & The Adjuster
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)

# 2. The Training Loop
epochs = 5
print(f"Starting Training on {device}...")

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        # Move batch to GPU
        images, labels = images.to(device), labels.to(device)

        # The 4-step Learning Cycle
        optimizer.zero_grad()               # Clear old mistakes
        outputs = model(images)            # Make a guess
        loss = criterion(outputs, labels)  # Calculate the error
        loss.backward()                    # Trace the error back
        optimizer.step()                   # Adjust the weights

        running_loss += loss.item()

    # Report progress
    average_loss = running_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{epochs}] - Loss: {average_loss:.4f}")

print("🏁 Training Finished!")

Starting Training on cuda...
Epoch [1/5] - Loss: 1.5861
Epoch [2/5] - Loss: 0.5741
Epoch [3/5] - Loss: 0.3825
Epoch [4/5] - Loss: 0.2956
Epoch [5/5] - Loss: 0.2400
🏁 Training Finished!


In [12]:
# Save the 'Brain'
torch.save(model.state_dict(), 'plant_model.pth')

import json
with open('classes.json', 'w') as f:
    json.dump(full_datasets.classes, f)

print("Files saved! ")

Files saved! 


In [13]:
# Check the actual folder names in the directory
import os
actual_folders = sorted(os.listdir(base_path))
print(actual_folders)


['0', '1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '3', '30', '31', '32', '33', '34', '35', '36', '37', '4', '5', '6', '7', '8', '9']


In [15]:
# The Master List of PlantVillage Classes in order
plant_village_classes = [
    "Apple___Apple_scab", "Apple___Black_rot", "Apple___Cedar_apple_rust", "Apple___healthy",
    "Blueberry___healthy", "Cherry___Powdery_mildew", "Cherry___healthy",
    "Corn___Cercospora_leaf_spot Gray_leaf_spot", "Corn___Common_rust", "Corn___Northern_Leaf_Blight", "Corn___healthy",
    "Grape___Black_rot", "Grape___Esca_(Black_Measles)", "Grape___Leaf_blight_(Isariopsis_Leaf_Spot)", "Grape___healthy",
    "Orange___Haunglongbing_(Citrus_greening)", "Peach___Bacterial_spot", "Peach___healthy",
    "Pepper,_bell___Bacterial_spot", "Pepper,_bell___healthy",
    "Potato___Early_blight", "Potato___Late_blight", "Potato___healthy",
    "Raspberry___healthy", "Soybean___healthy", "Squash___Powdery_mildew",
    "Strawberry___Leaf_scorch", "Strawberry___healthy",
    "Tomato___Bacterial_spot", "Tomato___Early_blight", "Tomato___Late_blight", "Tomato___Leaf_Mold",
    "Tomato___Septoria_leaf_spot", "Tomato___Spider_mites Two-spotted_spider_mite", "Tomato___Target_Spot",
    "Tomato___Tomato_Yellow_Leaf_Curl_Virus", "Tomato___Tomato_mosaic_virus", "Tomato___healthy"
]

# JSON file
import json
with open('classes.json', 'w') as f:
    json.dump(plant_village_classes, f)

print("✅ Success!  ")

✅ Success!  
